# 06 二阶数值微分

二阶数值微分常用于曲率、扩散项、Poisson 方程和波动方程离散化。它比一阶微分更敏感，因为典型公式要除以 $h^2$，函数值误差和数据噪声会被更强地放大。


In [ ]:
import math
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    differentiate_discrete,
    natural_cubic_spline_derivative,
    second_derivative_endpoint_three_point,
    second_derivative_five_point,
    second_derivative_three_point,
)


## 三点中心二阶差分

由 Taylor 展开相加可得

$$
f(x+h)+f(x-h)=2f(x)+h^2f''(x)+O(h^4),
$$

因此

$$
f''(x)\approx\frac{f(x-h)-2f(x)+f(x+h)}{h^2},
$$

截断误差为 $O(h^2)$。


In [ ]:
def teaching_second_center(f, x, h):
    return (f(x - h) - 2 * f(x) + f(x + h)) / h**2

x0 = 0.7
for h in [0.2, 0.1, 0.05, 0.025]:
    value = teaching_second_center(math.sin, x0, h)
    print(f"h={h:.3f}  value={value:.8f}  error={abs(value + math.sin(x0)):.3e}")


## 三点单边和五点中心二阶公式

端点附近可用三点单边二阶公式：

$$
f''(x)\approx\frac{f(x)-2f(x+h)+f(x+2h)}{h^2},
$$

它只是一阶精度。五点中心二阶公式为

$$
f''(x)\approx
\frac{-f(x-2h)+16f(x-h)-30f(x)+16f(x+h)-f(x+2h)}{12h^2},
$$

对光滑函数具有四阶截断误差。


In [ ]:
def teaching_second_five(f, x, h):
    return (-f(x - 2*h) + 16*f(x - h) - 30*f(x) + 16*f(x + h) - f(x + 2*h)) / (12*h**2)

h_values = 2.0 ** (-np.arange(3, 10, dtype=float))
err_three = []
err_five = []
for h in h_values:
    err_three.append(abs(teaching_second_center(math.sin, x0, h) + math.sin(x0)))
    err_five.append(abs(teaching_second_five(math.sin, x0, h) + math.sin(x0)))
print("三点中心误差：", err_three)
print("五点中心误差：", err_five)
print("正式五点实现：", float(second_derivative_five_point(np.sin, x0, 0.05)))
print("左端点三点实现：", float(second_derivative_endpoint_three_point(np.sin, 0.0, 0.05, side="left")))


## 一阶和二阶微分的噪声敏感性比较

一阶中心差分的噪声项大致除以 $h$，二阶中心差分的噪声项大致除以 $h^2$。因此二阶导数估计通常更不稳定。


In [ ]:
rng = np.random.default_rng(2026)
x = np.linspace(0.0, 2.0 * math.pi, 101)
clean = np.sin(x)
noisy = clean + 0.005 * rng.normal(size=x.size)
first_noisy = differentiate_discrete(x, noisy, derivative_order=1, stencil_size=5)
second_noisy = differentiate_discrete(x, noisy, derivative_order=2, stencil_size=5)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(x, np.cos(x), label="解析一阶导数")
axes[0].plot(x, first_noisy, "--", label="含噪声一阶估计")
axes[0].set_title("一阶导数")
axes[1].plot(x, -np.sin(x), label="解析二阶导数")
axes[1].plot(x, second_noisy, "--", label="含噪声二阶估计")
axes[1].set_title("二阶导数")
for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.show()


## 样条二阶微分

样条分段三次多项式的二阶导数是分段线性函数：

$$
S_i''(x)=2c_i+6d_i(x-x_i).
$$

这比直接二阶差分更平滑，但自然边界条件会直接影响端点附近的二阶导数。


In [ ]:
x_nodes = np.linspace(0.0, math.pi, 17)
y_nodes = np.sin(x_nodes)
x_eval = np.linspace(0.0, math.pi, 300)
spline_second = natural_cubic_spline_derivative(x_nodes, y_nodes, x_eval, derivative_order=2)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_eval, -np.sin(x_eval), label="解析二阶导数")
ax.plot(x_eval, spline_second, "--", label="自然三次样条二阶导数")
ax.scatter(x_nodes, natural_cubic_spline_derivative(x_nodes, y_nodes, x_nodes, derivative_order=2), s=20, label="节点值")
ax.set_title("样条二阶微分")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## 边界点处理

二阶导数在边界处不能使用中心公式。常见处理包括：使用单边公式、引入边界条件、使用样条导数，或在微分方程离散化中把边界值作为已知条件。选择方式取决于具体问题，而不是单纯追求公式阶数。


In [ ]:
h = 0.05
left_value = float(second_derivative_endpoint_three_point(np.exp, 0.0, h, side="left"))
center_value = float(second_derivative_three_point(np.exp, h, h))
print("左端点三点单边：", left_value, "误差", abs(left_value - 1.0))
print("内部三点中心：", center_value, "误差", abs(center_value - math.exp(h)))


## 小结

二阶微分公式和一阶公式一样可以从 Taylor 展开或局部插值多项式得到。不同之处在于误差放大更强，边界处理更敏感，含噪声数据上更容易出现大幅振荡。因此二阶导数实验必须同时观察截断误差、舍入误差和数据噪声。
